# Enhanced S3 to COG Converter with Automatic AWS Authentication

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading

Author: Kyle Lesinger (Enhanced version)

In [2]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
import fsspec
from rasterio.warp import calculate_default_transform, reproject, Resampling
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3


In [ ]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify,
    convert_to_proper_CRS_and_cogify_chunked
)

print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [ ]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [ ]:
EVENT_NAME = '202409_Hurricane_Helene'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'sentinel1'      #find the name within drcs_activations OLD Directory (see link above)

RENAME_PRODUCT = 'Sentinel-1'   #choose from LIST of 2nd level directories (see above list)

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory
DIRECTORY_NEW = f'{DIR_NEW_BASE}/{RENAME_PRODUCT}'

## Initialize AWS S3 Client with automatic credential detection

In [ ]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

## Load TIF Files from DRCS Data
### This may assist with diagnosing any issues that occur if no files are found in the code block above

This cell loads the pre-analyzed DRCS activation data from `drcs_activations_tif_files.json` which contains a complete inventory of all .tif files in the NASA Disasters S3 bucket.

The code will:
1. Load the JSON file containing the file inventory
2. Parse the `PATH_OLD` variable to find the corresponding directory
3. Extract all .tif filenames from that directory
4. Store them in `files_to_process` for later use

In [ ]:
# # Load the pre-analyzed DRCS TIF files data using imported functions
# # The JSON path is relative to the notebook location
# json_path = Path('../../s3-crawler/drcs_activations_tif_files.json')

# # Load DRCS data
# drcs_data = load_drcs_data(json_path)

# if drcs_data:
#     # Get TIF files from the specified PATH_OLD using the imported function
#     tif_files = get_tif_files_from_path(PATH_OLD, drcs_data, DIR_OLD_BASE)
    
#     if tif_files:
#         print(f"\n📁 Found {len(tif_files)} .tif files in {PATH_OLD}:")
#         print("\nFirst 10 files:")
#         for i, file in enumerate(tif_files[:10], 1):
#             print(f"  {i:2d}. {file}")
#         if len(tif_files) > 10:
#             print(f"  ... and {len(tif_files) - 10} more files")
        
#         # Get files with full paths using the imported function
#         files_to_process = get_files_with_full_paths(PATH_OLD, drcs_data, DIR_OLD_BASE, json_path)
#         print(f"\n✅ Files ready for processing. Stored in 'files_to_process' variable.")
#     else:
#         print(f"\n❌ No files found. Please check the PATH_OLD variable.")
#         files_to_process = []
# else:
#     print(f"\n❌ Could not load DRCS data.")
#     files_to_process = []

# files_to_process

In [ ]:
# # Example: List available activation events using the imported function
# print("📂 Available activation events in DRCS data:")
# events = list_available_directories('drcs_activations', drcs_data, json_path)

# # Show first 10 events
# for event in events[:10]:
#     print(f"  - {event}")
# if len(events) > 10:
#     print(f"  ... and {len(events) - 10} more events")

# # Example: List subdirectories for a specific event
# print(f"\n📁 Subdirectories in {EVENT_NAME}:")
# subdirs = list_available_directories(f'drcs_activations/{EVENT_NAME}', drcs_data, json_path)
# for subdir in subdirs:
#     print(f"  - {subdir}")

# For these we can see three different types of files

1. WM = water mask
2. rgb = red green blue
3. WM_diff = water mask difference between dates

### We are going to need 2 different directories for these!!!

We will keep WaterMask (WM) and rgb as separate directories

In [ ]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

water_mask = [f for f in keys if "_WM.tif" in f]
rgb = [f for f in keys if "rgb.tif" in f]
water_mask_diff = [f for f in keys if "WM_diff.tif" in f]

In [ ]:
rgb

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

In [ ]:
config_WM = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/WM",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_rgb = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/rgb", #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

## Configure bucket and paths (no need to create session manually)

In [ ]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }




In [ ]:
# Use the function with config_WM
wm_bucket = return_bucket_info(config_WM)
rgb_bucket = return_bucket_info(config_rgb)

In [ ]:
def convert_sentinel_datetime(datetime_str):
    """
    Convert Sentinel datetime format to ISO 8601 format with UTC timezone.
    
    Args:
        datetime_str: String like '20240430T002653'
    
    Returns:
        String like '2024-04-30T00:26:53Z'
    """
    # Extract components
    year = datetime_str[0:4]
    month = datetime_str[4:6]
    day = datetime_str[6:8]
    hour = datetime_str[9:11]
    minute = datetime_str[11:13]
    second = datetime_str[13:15]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}T{hour}:{minute}:{second}Z"

# Test
datetime_str = '20240430T002653'
result = convert_sentinel_datetime(datetime_str)
print(result)  # 2024-04-30T00:26:53Z

In [ ]:
# f'{EVENT_NAME}_{"_".join(fsplit[0:2])}_{"_".join(fsplit[3:8])}_{convert_sentinel_datetime(fsplit[2])}.tif'

In [ ]:
# Define COG profile for rasterio
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

## Define COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with proper CRS and caching.

In [ ]:
# Check current cache status using the imported function
check_cache_status()

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

# Process RGB files

In [ ]:
# NOTE for the diff files, we need to add diff at the 1st date before it
# Otherwise VEDA will think that the first date is the most important

# Example S1_diff20240430_20240507_WM.tif

In [ ]:
# Define filename creator functions for different file types

def create_cog_filename_WM(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Check if it's a diff file
    if "WM_diff" in f:
        # For diff files: S1_20240430_20240507_WM_diff.tif
        # Need to add "diff" before the first date
        # Result: S1_diff20240430_20240507_WM.tif
        return f'{EVENT_NAME}_S1_WM_diff{fsplit[1]}_{fsplit[2]}.tif'
    else:
        # Regular WM files
        cog_filename = f'{EVENT_NAME}_{"_".join(fsplit[0:2])}_{"_".join(fsplit[3:8])}_WM_{convert_sentinel_datetime(fsplit[2])}.tif'
        return cog_filename

def create_cog_filename_rgb(f, EVENT_NAME):
    """Create COG filename for RGB files, handling both simple and complex formats."""
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Check if it's a simple format (S1A_YYYYMMDD_rgb) or has _N_ variant
    if len(fsplit) <= 4 and fsplit[0] == 'S1A':
        # Simple format
        if len(fsplit) == 3:  # S1A_20240914_rgb
            date_str = fsplit[1]
            # Format date from YYYYMMDD to YYYY-MM-DD
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
            cog_filename = f'{EVENT_NAME}_S1A_rgb_{formatted_date}.tif'
        elif len(fsplit) == 4 and fsplit[2] == 'N':  # S1A_20240926_N_rgb
            date_str = fsplit[1]
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
            cog_filename = f'{EVENT_NAME}_S1A_N_rgb_{formatted_date}.tif'
        else:
            # Fallback
            cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    # Complex format with timestamp (S1A_IW_YYYYMMDDTHHMMSS_...)
    elif len(fsplit) > 4 and 'T' in fsplit[2]:
        # Extract parts and rebuild with rgb indicator
        cog_filename = f'{EVENT_NAME}_{"_".join(fsplit[0:2])}_{"_".join(fsplit[3:8])}_rgb_{convert_sentinel_datetime(fsplit[2])}.tif'
    
    else:
        # Fallback for any other format
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

def create_cog_filename_diff(f, EVENT_NAME):
    """Create COG filename for water mask diff files."""
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # For diff files: S1_20240430_20240507_WM_diff.tif
    # Need to add "diff" before the first date
    # Result: 202405_Flood_TX_S1_diff20240430_20240507_WM.tif
    return f'{EVENT_NAME}_S1_WM_diff{fsplit[1]}_{fsplit[2]}.tif'

# Test functions
print("Testing WM filename:")
test_wm = create_cog_filename_WM('drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_WM.tif', EVENT_NAME)
print(f"  {test_wm}")

print("\nTesting RGB filename:")
test_rgb = create_cog_filename_rgb('drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_rgb.tif', EVENT_NAME)
print(f"  {test_rgb}")

In [ ]:
rgb

In [ ]:
#First rgb

for idx,i in enumerate(rgb):
    test_rgb = create_cog_filename_rgb(rgb[idx], EVENT_NAME)
    print(f"  {test_rgb}")

In [ ]:
#Completed

# # Process water mask files
# if water_mask:
#     print("\n" + "="*50)
#     print("🌊 Processing Water Mask Files")
#     print("="*50)
#     # Initialize combined results DataFrame
#     all_files_processed = pd.DataFrame()
    
#     wm_results = process_file_batch(
#         file_list=water_mask,
#         s3_client=s3_client,
#         config=config_WM,
#         filename_creator_func=create_cog_filename_WM,
#         processing_func=convert_to_proper_CRS_and_cogify,
#         event_name=EVENT_NAME,
#         save_metadata=True,
#         save_csv=True,
#         verbose=True
#     )
#     all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)
    
#     # Print overall summary
#     print_batch_summary(all_files_processed)


In [ ]:

# Process RGB files
if rgb:
    print("\n" + "="*50)
    print("🎨 Processing RGB Files")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    rgb_results = process_file_batch(
        file_list=rgb,
        s3_client=s3_client,
        config=config_rgb,
        filename_creator_func=create_cog_filename_rgb,
        processing_func=convert_to_proper_CRS_and_cogify,
        event_name=EVENT_NAME,
        BUCKET=BUCKET,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, rgb_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)


In [ ]:
# Display final results (for a single instance)
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination. 
